# Setup Secret Scope
1. GET API URL and Token
2. Create a Secret Scope
3. Add Secrets to the scope

### Step1 - GET API URL and Token

In [0]:
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()

api_url = ctx.apiUrl().getOrElse(None)
api_token = ctx.apiToken().getOrElse(None)


### Step2 - Create a Secret Scope

In [0]:
#Define Connection Variables
jdbc_hostname = "neo-bank-server.database.windows.net"
jdbc_port = 1433
jdbc_database = "neo_bank_db"
jdbc_username = "[database user name goes here]"
jdbc_password = "[database passoword goes here]"  
driver = "com.microsoft.sqlserver.jdbc.SQLServerDriver"

secret_scope_name = "neobank-scope"
secret_key_name = "neobank-jdbc-connection-json"

In [0]:
import requests
import json

#configuration
DATABRICKS_HOST = api_url
DATABRICKS_TOKEN = api_token

url = f"{DATABRICKS_HOST}/api/2.0/secrets/scopes/create"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}


payload = {
        "scope": secret_scope_name,
        "scope_backend_type": "DATABRICKS"
    }

response = requests.post(url, headers=headers, data=json.dumps(payload))

if response.status_code == 200:
    print(f"Secret scope '{scope_name}' created successfully.")
else:
    print("Failed to create secret scope.")
    print("Status Code:", response.status_code)
    print("Response:", response.text)

### Step3 - Add Secrets to the scope

In [0]:
url = f"{DATABRICKS_HOST}/api/2.0/secrets/put"

connection_config = {
    "jdbc_hostname" : jdbc_hostname,
    "jdbc_port" : jdbc_port,
    "jdbc_database" : jdbc_database, 
    "jdbc_username" : jdbc_username, 
    "jdbc_password" : jdbc_password, 
    "driver" : driver
}
headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

payload = {
    "scope": secret_scope_name,
    "key": secret_key_name,
    "string_value": json.dumps(connection_config)
}

response = requests.post(url, headers=headers, data=json.dumps(payload))

if response.status_code == 200:
    print(f"Secret '{secret_key_name}' created successfully in scope '{secret_scope_name}'.")
else:
    print("Failed to create secret.")
    print("Status:", response.status_code)
    print("Response:", response.text)

In [0]:
try:
    secret_json = dbutils.secrets.get(
        scope=secret_scope_name,
        key=secret_key_name
    )
    
    print("Secret retrieved successfully.")
    
    config = json.loads(retrieved_json)
    print("Parsed JSON:")
    print(config)
    
except Exception as e:
    print("Secret verification failed:")
    print(str(e))

